<a href="https://colab.research.google.com/github/ozhao1323/ECON3916-Statistical-and-Machine-Learning/blob/main/Lab%2013/%5BLab_13%5D_Hedonic_Pricing_and_the_FWL_Theorem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Step 1: Ingestion and Naive Model
url = 'https://raw.githubusercontent.com/ozhao1323/ECON3916-Statistical-and-Machine-Learning/refs/heads/main/Data/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)

df

,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54
...,...,...,...
995,87.7,10.1,932592.35
996,21.2,91.8,412741.12
997,96.5,14.5,880901.56
998,20.1,95.1,396659.79


In [6]:
naive_model = smf.ols("Sale_Price ~ Property_Age",data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        19:53:32   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [7]:
# Step 2: The Multivariate Model
multi_model = smf.ols("Sale_Price ~ Property_Age + Distance_to_Tech_Hub", data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        19:57:41   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [13]:
# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid


In [12]:
# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid


In [14]:
# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])


FWL Isolated Age Coefficient: -2063.129216802138


In [16]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

# ── 1. SYNTHETIC DATA (mirrors the HTML simulation) ───────────────────────────
np.random.seed(42)
N = 120

# Independent variables
property_age      = np.random.uniform(2, 50, N)       # years
distance_to_hub   = np.random.uniform(0.5, 20, N)     # miles

# True DGP: Sale_Price = 450000 - 3500*Age - 8000*Distance + noise
sale_price = (450_000
              - 3_500 * property_age
              - 8_000 * distance_to_hub
              + np.random.normal(0, 35_000, N))

df = pd.DataFrame({
    'Sale_Price':        sale_price,
    'Property_Age':      property_age,
    'Distance_to_Hub':   distance_to_hub
})

# ── 2. OLS REGRESSION VIA STATSMODELS ────────────────────────────────────────
# Add a constant column so statsmodels estimates the intercept (β₀)
X = sm.add_constant(df[['Property_Age', 'Distance_to_Hub']])
y = df['Sale_Price']

model  = sm.OLS(y, X)
result = model.fit()
print(result.summary())

# ── 3. EXTRACT COEFFICIENTS FROM STATSMODELS RESULTS ─────────────────────────
# result.params is a pandas Series indexed by column name.
# Index 0 → intercept (β₀), 1 → Property_Age (β₁), 2 → Distance_to_Hub (β₂)
B0 = result.params['const']            # intercept
B1 = result.params['Property_Age']     # partial slope: age effect holding distance fixed
B2 = result.params['Distance_to_Hub']  # partial slope: distance effect holding age fixed

print(f"\nExtracted Coefficients:")
print(f"  β₀ (Intercept)       = ${B0:,.0f}")
print(f"  β₁ (Property_Age)    = ${B1:,.0f} per year")
print(f"  β₂ (Distance_to_Hub) = ${B2:,.0f} per mile")
print(f"  R²                   = {result.rsquared:.4f}")

# ── 4. MESHGRID FOR THE REGRESSION SURFACE ───────────────────────────────────
# Step 1: create two 1-D arrays spanning the observed range of each predictor.
#         These become the X and Y axes of the surface grid.
age_grid  = np.linspace(property_age.min()    - 1,
                        property_age.max()    + 1, 30)   # 30 evenly-spaced age values

dist_grid = np.linspace(distance_to_hub.min() - 0.5,
                        distance_to_hub.max() + 0.5, 30) # 30 evenly-spaced distance values

# Step 2: np.meshgrid expands those two 1-D arrays into two 2-D matrices (A, D)
#         so that every (A[i,j], D[i,j]) pair covers the full Cartesian grid.
A, D = np.meshgrid(age_grid, dist_grid)
#   A.shape == D.shape == (30, 30)

# Step 3: evaluate the fitted plane equation at every grid cell.
#         This is the 2-D hyperplane: Ŷ = β₀ + β₁·Age + β₂·Distance
Z = B0 + B1 * A + B2 * D
#   Z[i,j] is the model's predicted Sale_Price at age_grid[j], dist_grid[i]

# ── 5. RESIDUALS (for colored stems) ─────────────────────────────────────────
y_hat  = result.fittedvalues.values   # in-sample predictions for all 120 obs
resids = y.values - y_hat             # positive = above plane, negative = below

# ── 6. BUILD PLOTLY 3-D FIGURE ───────────────────────────────────────────────
fig = go.Figure()

# ── 6a. OLS Fitted Surface ────────────────────────────────────────────────────
fig.add_trace(go.Surface(
    x=age_grid,          # 1-D array → axis labels for columns of Z
    y=dist_grid,         # 1-D array → axis labels for rows of Z
    z=Z,                 # 2-D array of predicted prices across the grid
    name='OLS Fitted Plane',
    opacity=0.55,
    colorscale=[
        [0.0, 'rgba(100, 40, 180, 0.9)'],
        [0.5, 'rgba(130, 80, 220, 0.7)'],
        [1.0, 'rgba(180,130,255, 0.5)']
    ],
    showscale=False,
    contours=dict(
        x=dict(show=True, color='rgba(180,130,255,0.2)', width=1),
        y=dict(show=True, color='rgba(180,130,255,0.2)', width=1)
    ),
    hovertemplate=(
        'Age: %{x:.1f} yr<br>'
        'Dist: %{y:.1f} mi<br>'
        'Ŷ: $%{z:,.0f}<extra>OLS Plane</extra>'
    )
))

# ── 6b. Residual Stems ────────────────────────────────────────────────────────
# For each observation, draw a vertical line from the actual price to the fitted price.
# We interleave [actual, fitted, None] triples to get separate line segments in one trace.
stem_x, stem_y, stem_z, stem_color = [], [], [], []

for i in range(N):
    stem_x  += [property_age[i],    property_age[i],    None]
    stem_y  += [distance_to_hub[i], distance_to_hub[i], None]
    stem_z  += [sale_price[i],      y_hat[i],           None]
    c = '#FF6B6B' if resids[i] >= 0 else '#6BFFB8'   # red = over, green = under
    stem_color += [c, c, '#000000']

fig.add_trace(go.Scatter3d(
    x=stem_x, y=stem_y, z=stem_z,
    mode='lines',
    name='Residuals',
    line=dict(color=stem_color, width=1.5),
    opacity=0.5,
    hoverinfo='skip'
))

# ── 6c. Observed Data Points ──────────────────────────────────────────────────
fig.add_trace(go.Scatter3d(
    x=property_age,
    y=distance_to_hub,
    z=sale_price,
    mode='markers',
    name='Observed Data',
    marker=dict(
        size=5,
        color=sale_price,
        colorscale=[[0,'#00d4ff'],[0.5,'#7DF9FF'],[1,'#ffffff']],
        showscale=False,
        opacity=0.85,
        line=dict(color='rgba(125,249,255,0.25)', width=0.5)
    ),
    hovertemplate=(
        'Age: %{x:.1f} yr<br>'
        'Dist: %{y:.1f} mi<br>'
        'Price: $%{z:,.0f}<extra>Observed</extra>'
    )
))

# ── 7. LAYOUT ─────────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=(f'OLS Hyperplane  |  Ŷ = {B0/1000:.0f}K '
              f'{"+" if B1>=0 else ""}{B1/1000:.2f}K·Age '
              f'{"+" if B2>=0 else ""}{B2/1000:.2f}K·Dist  '
              f'|  R² = {result.rsquared:.4f}'),
        font=dict(size=13, color='#e8eaf0', family='monospace'),
        x=0.5
    ),
    paper_bgcolor='#07080d',
    plot_bgcolor ='#07080d',
    scene=dict(
        bgcolor='#07080d',
        xaxis=dict(
            title='Property Age (years)',
            titlefont=dict(color='#FF6B6B', size=11),
            tickfont=dict(color='#5a6070', size=9),
            gridcolor='#1e2130', showbackground=False
        ),
        yaxis=dict(
            title='Distance to Tech Hub (miles)',
            titlefont=dict(color='#C77DFF', size=11),
            tickfont=dict(color='#5a6070', size=9),
            gridcolor='#1e2130', showbackground=False
        ),
        zaxis=dict(
            title='Sale Price ($)',
            titlefont=dict(color='#7DF9FF', size=11),
            tickfont=dict(color='#5a6070', size=9),
            tickprefix='$',
            gridcolor='#1e2130', showbackground=False
        ),
        camera=dict(eye=dict(x=1.6, y=-1.6, z=0.9))
    ),
    legend=dict(
        font=dict(color='#9aa0b0', size=10, family='monospace'),
        bgcolor='rgba(14,16,24,0.85)',
        bordercolor='#1e2130', borderwidth=1
    ),
    margin=dict(l=0, r=0, t=50, b=0),
    hoverlabel=dict(
        bgcolor='#0e1018', bordercolor='#1e2130',
        font=dict(color='#e8eaf0', family='monospace', size=11)
    )
)

fig.show()

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.760
Model:                            OLS   Adj. R-squared:                  0.756
Method:                 Least Squares   F-statistic:                     185.0
Date:                Wed, 18 Mar 2026   Prob (F-statistic):           5.86e-37
Time:                        20:20:40   Log-Likelihood:                -1427.3
No. Observations:                 120   AIC:                             2861.
Df Residuals:                     117   BIC:                             2869.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const            4.548e+05   9454.850     